# APTOS 2019 — Test Set Predictions

Generate a Kaggle-format submission CSV (`id_code`, `diagnosis`) from the
trained checkpoint.  Uses the same data pipeline as training:

1. Ben Graham preprocessing (circular crop + local colour normalisation)
2. Validation transform (Resize 512 × 512 + ImageNet normalisation)
3. MC Dropout inference (T stochastic forward passes → predictive mean)
4. `argmax(mean_probs)` → predicted DR grade

In [ ]:
import sys
from pathlib import Path

# Ensure src/ is importable regardless of notebook CWD
PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / "src").is_dir():
    SRC_DIR = PROJECT_ROOT / "src"
elif (PROJECT_ROOT.parent / "src").is_dir():
    SRC_DIR = PROJECT_ROOT.parent / "src"
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise RuntimeError("Cannot locate src/ directory.")

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm

from config import (
    APTOS_DIR, IMAGE_SIZE, IMAGENET_MEAN, IMAGENET_STD,
    BATCH_SIZE, NUM_WORKERS, NUM_CLASSES, MC_DROPOUT_RATE,
    MC_INFERENCE_PASSES, USE_AMP, RANDOM_SEED,
    CHECKPOINT_DIR, RESULTS_DIR, DR_GRADES,
    seed_everything, setup_directories,
)
from model import create_model, create_baseline_model
from dataset import DRDataset, get_val_transform

seed_everything(RANDOM_SEED)
setup_directories()
print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}")

## Configuration

Change the settings below to switch between models or adjust MC passes.

In [ ]:
# ── Paths ──
TEST_CSV = APTOS_DIR / "test.csv"
TEST_IMAGES_DIR = APTOS_DIR / "test_images"

# ── Model selection ──
# Options: "baseline" (best fold-0 QWK 0.9088) or "cbam"
MODEL_TYPE = "baseline"

# Checkpoint — resolved via glob pattern (handles timestamped names)
import glob as _glob
import os as _os

_CHECKPOINT_PATTERNS = {
    "baseline": str(CHECKPOINT_DIR / "baseline_resnet50*fold0_best.pth"),
    "cbam":     str(CHECKPOINT_DIR / "cbam_resnet50*fold0_best.pth"),
}

_pattern = _CHECKPOINT_PATTERNS[MODEL_TYPE]
_matches = sorted(_glob.glob(_pattern), key=_os.path.getmtime, reverse=True)
if not _matches:
    raise FileNotFoundError(f"No checkpoint found for pattern: {_pattern}")

CHECKPOINT_PATH = Path(_matches[0])
print(f"Checkpoint: {CHECKPOINT_PATH.name}")

# ── Inference ──
MC_PASSES = MC_INFERENCE_PASSES   # T = 20 stochastic forward passes
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device:     {DEVICE}")
print(f"MC passes:  {MC_PASSES}")
print(f"Model:      {MODEL_TYPE}")

## Load Test Data

The APTOS test CSV contains only `id_code` (no labels — Kaggle holdout).
A dummy `diagnosis` column is added so `DRDataset` can be reused as-is.

In [ ]:
test_df = pd.read_csv(TEST_CSV)
print(f"Test images: {len(test_df)}")
print(test_df.head())

# DRDataset expects a "diagnosis" column — add a placeholder
test_df["diagnosis"] = -1

## Dataset & DataLoader

Uses the **validation transform** (Resize + ImageNet normalise — no augmentation)
and Ben Graham preprocessing, identical to the evaluation pipeline.

In [ ]:
test_dataset = DRDataset(
    df=test_df,
    image_dir=str(TEST_IMAGES_DIR),
    transform=get_val_transform(IMAGE_SIZE),
    preprocess=True,
    target_size=IMAGE_SIZE,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print(f"Dataset size: {len(test_dataset)}")
print(f"Batches:      {len(test_loader)}")

## Load Trained Model

In [ ]:
if MODEL_TYPE == "baseline":
    model = create_baseline_model(
        num_classes=NUM_CLASSES,
        dropout_rate=MC_DROPOUT_RATE,
        pretrained=False,
    )
else:
    model = create_model(
        num_classes=NUM_CLASSES,
        dropout_rate=MC_DROPOUT_RATE,
        pretrained=False,
    )

ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model = model.to(DEVICE)
model.eval()

epoch = ckpt.get("epoch", "?")
best_kappa = ckpt.get("best_kappa", "?")
print(f"Loaded {MODEL_TYPE.upper()} checkpoint")
print(f"  Trained epoch:  {epoch}")
print(f"  Best val QWK:   {best_kappa}")

## MC Dropout Inference

For each test image, perform **T** stochastic forward passes through the
model (MC Dropout stays active in `eval()` mode).  The predictive mean
across all passes is used to derive the final class prediction.

```
For each image x:
  ŷ₁, ŷ₂, …, ŷ_T  (softmax probability vectors)

  Predictive mean:    ȳ = (1/T) Σ ŷ_t
  Predicted class:    argmax(ȳ)
  Predictive entropy: H(ȳ) = −Σ ȳ_c · log(ȳ_c)
  Confidence:         max(ȳ)
```

In [ ]:
@torch.no_grad()
def predict_mc_dropout(
    model: torch.nn.Module,
    dataloader: DataLoader,
    device: torch.device,
    n_passes: int = MC_PASSES,
) -> dict:
    """Run T stochastic forward passes on unlabelled test images.

    Args:
        model: Trained model with MCDropout layers.
        dataloader: Test DataLoader (yields image, dummy_label).
        device: Compute device.
        n_passes: Number of MC forward passes T.

    Returns:
        Dictionary with arrays:
          - mean_probs:   (N, C) predictive mean distribution
          - predictions:  (N,)   argmax of mean_probs
          - entropy:      (N,)   predictive entropy
          - confidence:   (N,)   max of mean_probs
    """
    model.eval()
    amp_enabled = USE_AMP and device.type == "cuda"
    all_stacked = []

    pbar = tqdm(dataloader, desc=f"MC Inference (T={n_passes})")
    for images, _ in pbar:
        images = images.to(device, non_blocking=True)

        # T stochastic forward passes
        pass_probs = []
        for _ in range(n_passes):
            with torch.amp.autocast(
                device_type=device.type, enabled=amp_enabled,
            ):
                logits = model(images)
            probs = F.softmax(logits.float(), dim=1)  # [B, C]
            pass_probs.append(probs.cpu())

        stacked = torch.stack(pass_probs, dim=0).permute(1, 0, 2)  # [B, T, C]
        all_stacked.append(stacked)

    # Concatenate all batches → (N, T, C)
    all_stacked = torch.cat(all_stacked, dim=0)

    # Predictive mean
    mean_probs = all_stacked.mean(dim=1)  # [N, C]

    # Predictive entropy: H = −Σ p̄_c · log(p̄_c)
    entropy = -(mean_probs * torch.log(mean_probs.clamp(min=1e-10))).sum(dim=1)

    confidence, predictions = mean_probs.max(dim=1)

    return {
        "mean_probs":  mean_probs.numpy(),
        "predictions": predictions.numpy(),
        "entropy":     entropy.numpy(),
        "confidence":  confidence.numpy(),
    }


results = predict_mc_dropout(model, test_loader, DEVICE, n_passes=MC_PASSES)

## Generate Submission CSV

Kaggle format: `id_code,diagnosis`

In [ ]:
submission = pd.DataFrame({
    "id_code": test_df["id_code"].values,
    "diagnosis": results["predictions"],
})

# Save to outputs/results/
submission_path = RESULTS_DIR / f"submission_{MODEL_TYPE}.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission saved: {submission_path}")
print(f"Total predictions: {len(submission)}")
submission.head(10)

## Prediction Distribution & Uncertainty Summary

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── 1. Predicted grade distribution ──
grade_counts = submission["diagnosis"].value_counts().sort_index()
bar_colors = ["#2ecc71", "#f1c40f", "#e67e22", "#e74c3c", "#8e44ad"]
axes[0].bar(
    grade_counts.index,
    grade_counts.values,
    color=[bar_colors[g] for g in grade_counts.index],
    edgecolor="white",
)
axes[0].set_xlabel("Predicted DR Grade")
axes[0].set_ylabel("Count")
axes[0].set_title("Predicted Grade Distribution")
axes[0].set_xticks(range(5))
axes[0].set_xticklabels(
    [DR_GRADES[g] for g in range(5)], rotation=30, ha="right",
)

for grade, count in zip(grade_counts.index, grade_counts.values):
    axes[0].text(grade, count + 5, str(count), ha="center", fontweight="bold")

# ── 2. Entropy histogram ──
axes[1].hist(
    results["entropy"], bins=50,
    color="#3498db", edgecolor="white", alpha=0.8,
)
axes[1].axvline(
    np.percentile(results["entropy"], 90),
    color="red", linestyle="--", label="90th percentile",
)
axes[1].set_xlabel("Predictive Entropy")
axes[1].set_ylabel("Count")
axes[1].set_title("Uncertainty Distribution")
axes[1].legend()

# ── 3. Confidence histogram ──
axes[2].hist(
    results["confidence"], bins=50,
    color="#27ae60", edgecolor="white", alpha=0.8,
)
axes[2].set_xlabel("Confidence (max p\u0304)")
axes[2].set_ylabel("Count")
axes[2].set_title("Confidence Distribution")

plt.tight_layout()
plt.show()

# ── Summary stats ──
print(f"\n{'=' * 50}")
print(f"  TEST SET PREDICTION SUMMARY")
print(f"{'=' * 50}")
print(f"  Total images:      {len(submission)}")
print(f"  Mean entropy:      {results['entropy'].mean():.4f}")
print(f"  Median entropy:    {np.median(results['entropy']):.4f}")
print(f"  Mean confidence:   {results['confidence'].mean():.4f}")
print(f"  Median confidence: {np.median(results['confidence']):.4f}")
print()
print("  Grade distribution:")
for grade in range(5):
    count = (submission["diagnosis"] == grade).sum()
    pct = 100 * count / len(submission)
    print(f"    {grade} ({DR_GRADES[grade]:>17s}): {count:>5d}  ({pct:5.1f}%)")

# Flag high-uncertainty predictions
high_unc_threshold = np.percentile(results["entropy"], 90)
n_high_unc = (results["entropy"] > high_unc_threshold).sum()
print(f"\n  [!] {n_high_unc} images above 90th-percentile entropy")
print(f"      (>{high_unc_threshold:.3f}) — flagged for specialist review.")

## Detailed Results (with uncertainty scores)

An extended CSV with per-class probabilities, entropy, and confidence
for further analysis.

In [ ]:
detailed_rows = []
for i in range(len(test_df)):
    row = {
        "id_code":    test_df["id_code"].iloc[i],
        "diagnosis":  int(results["predictions"][i]),
        "confidence": float(results["confidence"][i]),
        "entropy":    float(results["entropy"][i]),
    }
    for c in range(NUM_CLASSES):
        row[f"p{c}_{DR_GRADES[c].replace(' ', '_')}"] = float(
            results["mean_probs"][i, c]
        )
    detailed_rows.append(row)

detailed_df = pd.DataFrame(detailed_rows)
detailed_path = RESULTS_DIR / f"submission_{MODEL_TYPE}_detailed.csv"
detailed_df.to_csv(detailed_path, index=False)
print(f"Detailed results saved: {detailed_path}")
detailed_df.head(10)